# 베이지안 최적화 실습

**Bayesian Optimization · BO**

예측값과 불확실성을 이용해 평가 비용이 큰 목표 함수의 좋은 조건을 찾는 방법.

소재 분야에서 이해하기: 실험 횟수를 아끼며 합성 온도와 조성을 선택한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [베이지안 능동학습 연구](https://www.nature.com/articles/s41467-020-19597-w)

## 1. 최적 조건 찾기

실험 1회가 비싼 상황에서 최댓값을 찾습니다. 무작위 탐색과 비교합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.stats import norm

def yield_curve(x):
    """가상 수율(%): 온도(0-1로 정규화)에 따라 두 개의 봉우리가 있습니다."""
    return 70 * np.exp(-((x - 0.3) ** 2) / 0.01) + 92 * np.exp(-((x - 0.75) ** 2) / 0.006)

grid = np.linspace(0, 1, 500)
plt.plot(grid, yield_curve(grid)); plt.xlabel('normalised temperature'); plt.ylabel('yield (%)'); plt.show()
print('참 최댓값 %.1f%% (x=%.3f)' % (yield_curve(grid).max(), grid[np.argmax(yield_curve(grid))]))

In [ ]:
def expected_improvement(mean, std, best):
    std = np.maximum(std, 1e-9)
    z = (mean - best) / std
    return (mean - best) * norm.cdf(z) + std * norm.pdf(z)

def optimise(budget=15, seed=1):
    local = np.random.default_rng(seed)
    x_seen = list(local.uniform(0, 1, 3))
    y_seen = [yield_curve(value) for value in x_seen]
    best_history = [max(y_seen)]
    for step in range(budget):
        model = GaussianProcessRegressor(kernel=ConstantKernel(50.0) * RBF(0.08),
                                         normalize_y=True, alpha=1e-6, random_state=0)
        model.fit(np.array(x_seen)[:, None], y_seen)
        mean, std = model.predict(grid[:, None], return_std=True)
        candidate = grid[int(np.argmax(expected_improvement(mean, std, max(y_seen))))]
        x_seen.append(float(candidate)); y_seen.append(float(yield_curve(candidate)))
        best_history.append(max(y_seen))
    return np.array(best_history), x_seen, y_seen

bo_history, x_seen, y_seen = optimise()
random_history = np.maximum.accumulate([yield_curve(v) for v in rng.uniform(0, 1, 18)])
plt.plot(bo_history, 'o-', label='bayesian optimisation')
plt.plot(random_history, 's-', label='random search')
plt.axhline(yield_curve(grid).max(), color='k', ls='--', label='true maximum')
plt.xlabel('experiments'); plt.ylabel('best yield so far (%)'); plt.legend(); plt.show()
print('같은 실험 횟수에서 최고 수율: BO %.1f%% / 무작위 %.1f%%' % (bo_history[-1], random_history[-1]))

In [ ]:
model = GaussianProcessRegressor(kernel=ConstantKernel(50.0) * RBF(0.08),
                                 normalize_y=True, alpha=1e-6, random_state=0).fit(np.array(x_seen)[:, None], y_seen)
mean, std = model.predict(grid[:, None], return_std=True)
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(grid, yield_curve(grid), 'k--', label='truth')
axes[0].plot(grid, mean, label='GP mean')
axes[0].fill_between(grid, mean - 2 * std, mean + 2 * std, alpha=0.2)
axes[0].scatter(x_seen, y_seen, c='red', s=20, zorder=5, label='experiments')
axes[0].legend(fontsize=8); axes[0].set_ylabel('yield (%)')
axes[1].plot(grid, expected_improvement(mean, std, max(y_seen)))
axes[1].set_ylabel('expected improvement'); axes[1].set_xlabel('normalised temperature')
plt.tight_layout(); plt.show()
print('실험이 몰린 곳에서는 획득 함수가 0에 가까워지고, 아직 볼 만한 곳이 남으면 값이 큽니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#bayesian-optimization)을 여세요.